In [ ]:
import torch
import torchvision.transforms as transforms
from torchvision.datasets import EMNIST
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.Grayscale(3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_dataset = EMNIST(root="./data", split="letters", train=True, download=True, transform=transform)
test_dataset = EMNIST(root="./data", split="letters", train=False, download=True, transform=transform)

num_classes = 26
print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")
print(f"Number of classes: {num_classes}")

In [ ]:
letters = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

images, labels = next(iter(train_loader))

mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
images = (images * std + mean).clamp(0, 1)

fig, axes = plt.subplots(1, 6, figsize=(12, 3))
for i in range(6):
    img = images[i].permute(1, 2, 0)
    axes[i].imshow(img)
    axes[i].set_title(letters[labels[i] - 1])
    axes[i].axis("off")
plt.show()

In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_v2_s

# Write your code here
import torch.nn as nn
from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights

model = efficientnet_v2_s(weights=EfficientNet_V2_S_Weights.DEFAULT)

for p in model.features.parameters():
    p.requires_grad = False

in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, 26)

In [ ]:
# Write your code here
import torch

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for images, labels in loader:
        images = images.to(device)
        labels = (labels - 1).to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, correct / total

def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = (labels - 1).to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * images.size(0)
            _, preds = outputs.max(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, correct / total

In [ ]:
# Write your code here
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

train_subset = Subset(train_dataset, range(1000))
test_subset = Subset(test_dataset, range(500))

train_loader = DataLoader(train_subset, batch_size=32, shuffle=True, num_workers=0)
test_loader = DataLoader(test_subset, batch_size=32, shuffle=False, num_workers=0)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=1e-3)

epochs = 1
train_losses, val_losses = [], []
train_accs, val_accs = [], []

for e in range(epochs):
    tl, ta = train_one_epoch(model, train_loader, criterion, optimizer, device)
    vl, va = validate(model, test_loader, criterion, device)
    train_losses.append(tl)
    val_losses.append(vl)
    train_accs.append(ta)
    val_accs.append(va)
    print(f"Epoch {e+1}/{epochs} | train_loss={tl:.4f} train_acc={ta:.4f} | val_loss={vl:.4f} val_acc={va:.4f}")

plt.figure()
plt.plot(train_losses, marker='o')
plt.plot(val_losses, marker='o')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend(["Train Loss", "Val Loss"])
plt.show()

plt.figure()
plt.plot(train_accs, marker='o')
plt.plot(val_accs, marker='o')
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend(["Train Acc", "Val Acc"])
plt.show()

In [ ]:
# Write your code here
def validate_tta(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = (labels - 1).to(device)

            out1 = model(images)
            h_flipped = torch.flip(images, dims=[3])
            out2 = model(h_flipped)
            v_flipped = torch.flip(images, dims=[2])
            out3 = model(v_flipped)

            outputs = (out1 + out2 + out3) / 3.0
            loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)
            _, preds = outputs.max(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, correct / total